# Meta-SGD + SAM Transformer for daily Nifty 50 signals (Numin2)

This notebook holds the four configurations of the project's ablation study. Each code cell is
self-contained (imports, data pipeline, model, meta-training, allocator and Numin2 backtest) and
ends by running its own experiment, so the cells can be run independently. **Experiment 4 is the
final model.**

* **Data:** the cells read `../data/consolidated_daily_ohlcv.parquet` and
  `../data/consolidated_daily_returns.parquet`. See the repository README for how to download
  them with the `numin2` package.
* **Outputs:** the notebook was saved without outputs. The backtest results for these
  configurations are reported in `../docs/meta-learning-numin2-presentation.pdf` (slides 7-8).
* **Hardware:** the cells select `cuda:1` (or `cuda`) when a GPU is available; change the
  `device` line if your machine has a single GPU.

## Experiment 1: returns-only features (no OHLCV)

* Inputs: daily returns only, expanded to raw return, cross-sectional z-score, 60-day momentum
  and 20-day rolling volatility; 252-day windows.
* Meta-training: Meta-SGD (learned per-parameter inner learning rates) with SAM and an L1
  sparsity penalty on the predictions.
* Test time: online adaptation on the previous block, then the Absolute Conviction allocator
  (`top_k=2`, `enter_threshold=0.6`, `inertia=0.4`). Also prints win rate, max drawdown and Sortino.

In [ ]:
import math
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

from numin2 import Numin2API

# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')

# Setup device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Custom Dataloader Pipeline (Returns Only)
# ==========================================
class NuminReturnsDataset(Dataset):
    def __init__(self, returns_path, seq_len=252): 
        print("Loading Returns Parquet file...")
        df_returns = pd.read_parquet(returns_path)
        
        # Flatten Returns if it's a wide flat table
        if not isinstance(df_returns.columns, pd.MultiIndex) and 'asset' not in [str(c).lower() for c in df_returns.columns]:
            df_returns.index.name = 'date'
            df_returns = df_returns.reset_index().melt(id_vars=['date'], var_name='asset', value_name='returns')
        elif isinstance(df_returns.index, pd.MultiIndex):
            df_returns = df_returns.reset_index()

        # Standardize column names
        df_returns.columns = [str(c).lower() for c in df_returns.columns]
        
        print("Schema flattened. Executing sort...")
        df_returns.sort_values(by=['date', 'asset'], inplace=True)
        
        self.dates = df_returns['date'].unique()
        self.num_assets = len(df_returns['asset'].unique()) 
        
        # Only 1 feature column now (returns)
        self.data = np.zeros((len(self.dates), self.num_assets, 1))
        
        pivot_df = df_returns.pivot(index='date', columns='asset', values='returns').fillna(0)
        self.data[:, :, 0] = pivot_df.values
            
        self.data = torch.FloatTensor(self.data)
        self.seq_len = seq_len
        print(f"Data successfully compiled! Tensor Shape: {self.data.shape}")

    def __len__(self):
        return len(self.dates) - self.seq_len

    def __getitem__(self, idx):
        X = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len, :, 0]  # Target is the next day's return
        return X, y

def get_custom_dataloaders(returns_path, batch_size=128):
    full_dataset = NuminReturnsDataset(returns_path)
    
    # EXACT ORIGINAL SPLIT: 70% Train, 15% Val, 15% Test
    train_size = int(0.7 * len(full_dataset))
    val_size = int(0.15 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    
    train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
    val_dataset = torch.utils.data.Subset(full_dataset, range(train_size, train_size + val_size))
    test_dataset = torch.utils.data.Subset(full_dataset, range(train_size + val_size, len(full_dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

# ==========================================
# 2. Ablation Architecture (Returns Only)
# ==========================================
class ReturnsOnlyFeatureBlock(nn.Module):
    def __init__(self, num_assets=50):
        super(ReturnsOnlyFeatureBlock, self).__init__()
        self.num_assets = num_assets
        # 4 features purely derived from returns
        self.out_features = num_assets * 4 

    def forward(self, x):
        B, T, N, F_dim = x.shape
        R = x[..., 0]  # Only Returns are passed in
        
        # 1. Cross-Sectional Z-Score of Returns
        mean_r = R.mean(dim=-1, keepdim=True)
        std_r = R.std(dim=-1, keepdim=True) + 1e-6
        z_score = ((R - mean_r) / std_r).unsqueeze(-1)
        
        # 2. Macro Momentum (60d)
        cumsum_r = torch.cumsum(R, dim=1)
        mom_60d = torch.zeros_like(R)
        mom_60d[:, :60, :] = cumsum_r[:, :60, :]
        if T > 60:
            mom_60d[:, 60:, :] = cumsum_r[:, 60:, :] - cumsum_r[:, :-60, :]
        mom_60d = (mom_60d / 60.0).unsqueeze(-1) 
        
        # 3. Rolling Volatility Proxy (20d SMA of Absolute Returns)
        abs_r = torch.abs(R)
        cumsum_abs = torch.cumsum(abs_r, dim=1)
        roll_vol = torch.zeros_like(R)
        roll_vol[:, :20, :] = cumsum_abs[:, :20, :] / torch.arange(1, min(T, 20)+1, device=x.device).view(1, -1, 1)
        if T > 20:
            roll_vol[:, 20:, :] = (cumsum_abs[:, 20:, :] - cumsum_abs[:, :-20, :]) / 20.0
        roll_vol = roll_vol.unsqueeze(-1)
        
        # 4. Concatenate: Raw Returns, Z-Score, 60d Momentum, 20d Rolling Volatility
        out = torch.cat([R.unsqueeze(-1), z_score, mom_60d, roll_vol], dim=-1)
        return torch.nan_to_num(out.view(B, T, -1), nan=0.0)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.4, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MetaSGDTransformer(nn.Module):
    def __init__(self, num_assets=50, d_model=256, nhead=8, num_layers=4, dropout=0.3):
        super(MetaSGDTransformer, self).__init__()
        self.feature_block = ReturnsOnlyFeatureBlock(num_assets=num_assets)
        self.input_projection = nn.Linear(self.feature_block.out_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, 128), nn.LayerNorm(128), nn.GELU(), 
            nn.Dropout(dropout), nn.Linear(128, num_assets)
        )
        
        self.inner_lrs = nn.ParameterDict({
            name.replace('.', '_'): nn.Parameter(torch.ones_like(param) * 0.005)
            for name, param in self.named_parameters() if param.requires_grad
        })

    def forward(self, x):
        x = self.feature_block(x)
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        encoded = self.transformer(x)
        raw_alpha = self.output_projection(encoded[:, -1, :])
        return torch.tanh(raw_alpha)

# ==========================================
# 3. Meta-Optimizers & Losses
# ==========================================
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  
        self.base_optimizer.step()  
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]), p=2
        )
        return norm

def differentiable_sharpe_loss(predictions, targets, txn_cost=0.1):
    portfolio_returns = (predictions * targets).sum(dim=1) 
    cash_start = torch.zeros(1, predictions.size(1), device=predictions.device)
    prev_predictions = torch.cat([cash_start, predictions[:-1]], dim=0)
    
    turnover = torch.abs(predictions - prev_predictions).sum(dim=1)
    net_returns = portfolio_returns - (txn_cost * turnover)
    
    mean_return = net_returns.mean()
    std_return = net_returns.std() + 1e-6 
    return -((mean_return / std_return) * math.sqrt(252))

def train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.001):
    base_optimizer = torch.optim.AdamW
    meta_optimizer = SAM(model.parameters(), base_optimizer, lr=meta_lr, weight_decay=1e-4, rho=0.05)
    inner_criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(meta_epochs):
        epoch_loss = 0.0
        batches = list(train_loader)
        
        for i in range(0, len(batches) - 1, 2):
            X_adapt, y_adapt = batches[i][0].to(device), batches[i][1].to(device)
            X_eval, y_eval = batches[i+1][0].to(device), batches[i+1][1].to(device)
            
            y_adapt = torch.nan_to_num(y_adapt, nan=0.0)
            y_eval = torch.nan_to_num(y_eval, nan=0.0)
            
            # PASS 1
            preds = model(X_adapt)
            inner_loss = inner_criterion(preds, y_adapt)
            grads = torch.autograd.grad(inner_loss, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    adapted_weights[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
            
            meta_optimizer.zero_grad()
            eval_preds = functional_call(model, adapted_weights, X_eval) 
            base_sharpe_loss = differentiable_sharpe_loss(eval_preds, y_eval, txn_cost=0.1)
            
            sparsity_penalty = 0.5 * torch.mean(torch.abs(eval_preds)) 
            meta_loss = base_sharpe_loss + sparsity_penalty
            meta_loss.backward() 
            meta_optimizer.first_step(zero_grad=True)
            
            # PASS 2
            preds_adv = model(X_adapt)
            inner_loss_adv = inner_criterion(preds_adv, y_adapt)
            grads_adv = torch.autograd.grad(inner_loss_adv, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights_adv = {}
            for (name, param), grad in zip(model.named_parameters(), grads_adv):
                if grad is None:
                    adapted_weights_adv[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights_adv[name] = param - (model.inner_lrs[clean_name] * grad)
            
            eval_preds_perturbed = functional_call(model, adapted_weights_adv, X_eval)
            base_sharpe_loss_adv = differentiable_sharpe_loss(eval_preds_perturbed, y_eval, txn_cost=0.1)
            
            sparsity_penalty_adv = 0.5 * torch.mean(torch.abs(eval_preds_perturbed))
            meta_loss_perturbed = base_sharpe_loss_adv + sparsity_penalty_adv
            
            meta_loss_perturbed.backward()
            meta_optimizer.second_step(zero_grad=True)
            
            epoch_loss += meta_loss_perturbed.item()
            
        num_task_batches = len(batches) // 2
        avg_loss = (epoch_loss / num_task_batches) if num_task_batches > 0 else 0.0
        print(f"Meta-Epoch {epoch+1}/{meta_epochs} | Avg Worst-Case Sharpe Loss: {avg_loss:.4f}")
        
    return model

def allocate_portfolio_absolute(continuous_signals, prev_weights=None, top_k=2, enter_threshold=0.6, inertia=0.4):
    batch_size = continuous_signals.shape[0]
    num_assets = continuous_signals.shape[1]
    weights = torch.zeros_like(continuous_signals)
    
    if prev_weights is None:
        prev_weights = torch.zeros(num_assets, device=continuous_signals.device)
        
    for i in range(batch_size):
        alphas = continuous_signals[i] 
        adjusted_alphas = alphas + (prev_weights * inertia)
        
        long_scores, top_long_idx = torch.topk(adjusted_alphas, top_k)
        short_scores, top_short_idx = torch.topk(-adjusted_alphas, top_k)
        
        current_w = torch.zeros(num_assets, device=continuous_signals.device)
        
        for idx, score in zip(top_long_idx, long_scores):
            if score > enter_threshold: 
                current_w[idx] = 1.0
                
        for idx, score in zip(top_short_idx, short_scores):
            if score > enter_threshold: 
                current_w[idx] = -1.0

        weights[i] = current_w
        prev_weights = current_w 

    active_exposure = torch.abs(weights).sum(dim=1, keepdim=True)
    cash_column = torch.where(active_exposure == 0, 1.0, 0.0)
    
    return torch.cat((cash_column, weights), dim=1), prev_weights

# ==========================================
# 4. Elite Metrics Calculator
# ==========================================
def calculate_elite_metrics(results_dict):
    pnl_array = np.array(results_dict['incremental_pnl'])
    
    # Win Rate
    winning_days = np.sum(pnl_array > 0)
    total_active_days = np.sum(pnl_array != 0)
    win_rate = (winning_days / total_active_days) * 100 if total_active_days > 0 else 0
    
    # Maximum Drawdown
    cumulative_returns = np.cumsum(pnl_array)
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdowns = running_max - cumulative_returns
    max_drawdown = np.max(drawdowns)
    
    # Sortino Ratio
    mean_return = np.mean(pnl_array)
    downside_returns = pnl_array[pnl_array < 0]
    downside_std = np.std(downside_returns) if len(downside_returns) > 1 else 1e-6
    sortino_ratio = (mean_return / downside_std) * math.sqrt(252)
    
    print("\n" + "="*40)
    print("ELITE QUANT METRICS (Returns-Only Ablation)")
    print("="*40)
    print(f"Total Profit    : {results_dict['total_profit']:.4f}")
    print(f"Sharpe Ratio    : {results_dict['sharpe_ratio']:.4f}")
    print(f"Win Rate        : {win_rate:.2f}%")
    print(f"Max Drawdown    : {max_drawdown:.4f}")
    print(f"Sortino Ratio   : {sortino_ratio:.4f}")
    print("="*40)

# ==========================================
# 5. Execution Engine
# ==========================================
def execute_ablation_strategy(returns_path):
    print("Initializing Ablation Strategy Engine (70/15/15 Split)...")
    
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    
    client = Numin2API()
    
    print("Generating Custom Temporal Splits (70% Train, 15% Val, 15% Test)...")
    train_loader, val_loader, test_loader = get_custom_dataloaders(returns_path, batch_size=128)
    
    model = MetaSGDTransformer(
        num_assets=50, 
        d_model=256, 
        nhead=8, 
        num_layers=4, 
        dropout=0.3
    ).to(device)
    
    print(f"\n{'='*55}")
    print("--- Phase 1: Meta-Training ---")
    print(f"{'='*55}")
    model = train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.0003)
    
    print(f"\n{'='*55}")
    print("--- Phase 2: Out-of-Sample Testing (ONLINE ADAPTATION) ---")
    print(f"{'='*55}")
    
    all_test_positions = []
    all_test_targets = []
    current_portfolio_state = None 
    
    X_prev, y_prev = None, None
    inner_criterion = nn.MSELoss()
    
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        clean_y_batch = torch.nan_to_num(y_batch, nan=0.0).to(device)
        
        if X_prev is not None:
            model.train() 
            preds_prev = model(X_prev)
            loss_prev = inner_criterion(preds_prev, y_prev)
            
            grads = torch.autograd.grad(loss_prev, model.parameters(), retain_graph=False, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    adapted_weights[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
        else:
            adapted_weights = {name: param for name, param in model.named_parameters()}
            
        with torch.no_grad():
            model.eval()
            continuous_predictions = functional_call(model, adapted_weights, X_batch)
            
            discrete_batch_positions, current_portfolio_state = allocate_portfolio_absolute(
                continuous_predictions, 
                prev_weights=current_portfolio_state,
                top_k=2,
                enter_threshold=0.6, 
                inertia=0.4
            )
            
        nan_mask = torch.isnan(y_batch).to(device)
        discrete_batch_positions[:, 1:][nan_mask] = 0.0 
        
        all_test_positions.append(discrete_batch_positions.cpu())
        all_test_targets.append(clean_y_batch.cpu())
        
        X_prev = X_batch
        y_prev = clean_y_batch
        
    full_test_positions = torch.cat(all_test_positions, dim=0)
    full_test_targets = torch.cat(all_test_targets, dim=0)
    
    print(f">> Submitting {full_test_positions.shape[0]} trading days to Numin2 Backtester...")
    
    results = client.backtest_positions(
        positions=full_test_positions.numpy(), 
        targets=full_test_targets.numpy(), 
        txn_cost=0.1
    )
    
    # Calculate and print the final metrics for the report
    calculate_elite_metrics(results)
    
    return results

if __name__ == "__main__":
    RETURNS_PATH = '../data/consolidated_daily_returns.parquet'
    execute_ablation_strategy(RETURNS_PATH)

## Experiment 2: full OHLCV + Meta-SGD + SAM, strict discrete allocator

* Inputs: OHLCV + returns plus Garman-Klass volatility, relative volume, cross-sectional z-score
  and 5-day momentum; 60-day windows, batch size 32.
* Meta-training: Meta-SGD + SAM on the turnover-aware Sharpe loss (no sparsity penalty).
* Test time: no online adaptation; top-4 long / top-4 short with a 0.15 signal threshold.

In [ ]:
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F  # <--- Add this line right here
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

# --- API Imports ---
from numin2 import Numin2API, NuminDataset, get_range_dataloader, get_month_dataloader
from torch.func import functional_call
# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')
DATA_PATH = '/tmp/consolidated_daily_returns.parquet' 

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from torch.utils.data import Dataset, DataLoader

import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

from numin2 import Numin2API

# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Custom Dataloader Pipeline
# ==========================================
class NuminAdvancedDataset(Dataset):
    def __init__(self, ohlcv_path, returns_path, seq_len=60):
        print("Loading and aligning Parquet files...")
        df_ohlcv = pd.read_parquet(ohlcv_path)
        df_returns = pd.read_parquet(returns_path)
        
        # 1. Flatten OHLCV if it has MultiIndex Columns
        if isinstance(df_ohlcv.columns, pd.MultiIndex):
            df_ohlcv = df_ohlcv.stack(level=1, future_stack=True).reset_index()
            df_ohlcv.rename(columns={df_ohlcv.columns[0]: 'date', df_ohlcv.columns[1]: 'asset'}, inplace=True)
        elif isinstance(df_ohlcv.index, pd.MultiIndex):
            df_ohlcv = df_ohlcv.reset_index()
            
        # 2. Flatten Returns if it's a wide flat table
        if not isinstance(df_returns.columns, pd.MultiIndex) and 'asset' not in [str(c).lower() for c in df_returns.columns]:
            df_returns.index.name = 'date'
            df_returns = df_returns.reset_index().melt(id_vars=['date'], var_name='asset', value_name='returns')
        elif isinstance(df_returns.index, pd.MultiIndex):
            df_returns = df_returns.reset_index()

        # 3. Standardize column names
        df_ohlcv.columns = [str(c).lower() for c in df_ohlcv.columns]
        df_returns.columns = [str(c).lower() for c in df_returns.columns]
        
        if 'returns' in df_ohlcv.columns and 'returns' in df_returns.columns:
            df_ohlcv.drop(columns=['returns'], inplace=True)
            
        print("Schema flattened. Executing merge...")
        df_merged = pd.merge(df_ohlcv, df_returns, on=['date', 'asset'], how='inner')
        df_merged.sort_values(by=['date', 'asset'], inplace=True)
        
        self.dates = df_merged['date'].unique()
        self.num_assets = len(df_merged['asset'].unique()) 
        
        feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']
        
        self.data = np.zeros((len(self.dates), self.num_assets, 6))
        
        for i, f in enumerate(feature_cols):
            if f in df_merged.columns:
                pivot_df = df_merged.pivot(index='date', columns='asset', values=f).fillna(0)
                self.data[:, :, i] = pivot_df.values
            
        self.data = torch.FloatTensor(self.data)
        self.seq_len = seq_len
        print(f"Data successfully compiled! Tensor Shape: {self.data.shape}")

    def __len__(self):
        return len(self.dates) - self.seq_len

    def __getitem__(self, idx):
        X = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len, :, 5] 
        return X, y

def get_custom_dataloaders(ohlcv_path, returns_path, batch_size=32):
    full_dataset = NuminAdvancedDataset(ohlcv_path, returns_path)
    # Temporal splitting as required by the assignment
    train_size = int(0.7 * len(full_dataset))
    val_size = int(0.15 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    
    train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
    val_dataset = torch.utils.data.Subset(full_dataset, range(train_size, train_size + val_size))
    test_dataset = torch.utils.data.Subset(full_dataset, range(train_size + val_size, len(full_dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

# ==========================================
# 2. Advanced Architecture
# ==========================================
class AdvancedQuantFeatureBlock(nn.Module):
    def __init__(self, num_assets=50):
        super(AdvancedQuantFeatureBlock, self).__init__()
        self.num_assets = num_assets
        self.out_features = num_assets * 10

    def forward(self, x):
        B, T, N, F_dim = x.shape
        O, H, L, C = x[..., 0], x[..., 1], x[..., 2], x[..., 3]
        V, R = x[..., 4], x[..., 5]
        
        # 1. Garman-Klass Volatility Proxy (WITH NAN SAFEGUARD)
        log_hl = torch.log(H / (L + 1e-6) + 1e-6)
        log_co = torch.log(C / (O + 1e-6) + 1e-6)
        
        # THE FIX: Clamp the internal math to a minimum of 0.0 before the square root
        gk_internal = 0.5 * log_hl**2 - (2 * math.log(2) - 1) * log_co**2
        gk_vol = torch.sqrt(torch.clamp(gk_internal, min=0.0)).unsqueeze(-1)
        
        # 2. Relative Volume
        cumsum_v = torch.cumsum(V, dim=1)
        sma_v = torch.zeros_like(V)
        sma_v[:, :20, :] = cumsum_v[:, :20, :] / torch.arange(1, min(T, 20)+1, device=x.device).view(1, -1, 1)
        if T > 20:
            sma_v[:, 20:, :] = (cumsum_v[:, 20:, :] - cumsum_v[:, :-20, :]) / 20.0
        rel_vol = (V / (sma_v + 1e-6)).unsqueeze(-1)
        
        # 3. Cross-Sectional Z-Score
        mean_r = R.mean(dim=-1, keepdim=True)
        std_r = R.std(dim=-1, keepdim=True) + 1e-6
        z_score = ((R - mean_r) / std_r).unsqueeze(-1)
        
        # 4. 5-Day Momentum
        cumsum_r = torch.cumsum(R, dim=1)
        mom_5d = torch.zeros_like(R)
        mom_5d[:, :5, :] = cumsum_r[:, :5, :]
        if T > 5:
            mom_5d[:, 5:, :] = cumsum_r[:, 5:, :] - cumsum_r[:, :-5, :]
        mom_5d = mom_5d.unsqueeze(-1)
        
        out = torch.cat([x, gk_vol, rel_vol, z_score, mom_5d], dim=-1)
        return torch.nan_to_num(out.view(B, T, -1), nan=0.0)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.4, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MetaSGDTransformer(nn.Module):
    def __init__(self, num_assets=50, d_model=256, nhead=8, num_layers=4, dropout=0.3):
        super(MetaSGDTransformer, self).__init__()
        self.feature_block = AdvancedQuantFeatureBlock(num_assets=num_assets)
        self.input_projection = nn.Linear(self.feature_block.out_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, 128), nn.LayerNorm(128), nn.GELU(), 
            nn.Dropout(dropout), nn.Linear(128, num_assets)
        )
        
        self.inner_lrs = nn.ParameterDict({
            name.replace('.', '_'): nn.Parameter(torch.ones_like(param) * 0.005)
            for name, param in self.named_parameters() if param.requires_grad
        })

    def forward(self, x):
        x = self.feature_block(x)
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        encoded = self.transformer(x)
        raw_alpha = self.output_projection(encoded[:, -1, :])
        return torch.tanh(raw_alpha)

# ==========================================
# 3. Meta-Optimizers & Losses
# ==========================================
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  
        self.base_optimizer.step()  
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]),
            p=2
        )
        return norm

def differentiable_sharpe_loss(predictions, targets, txn_cost=0.1):
    portfolio_returns = (predictions * targets).sum(dim=1) 
    cash_start = torch.zeros(1, predictions.size(1), device=predictions.device)
    prev_predictions = torch.cat([cash_start, predictions[:-1]], dim=0)
    
    turnover = torch.abs(predictions - prev_predictions).sum(dim=1)
    net_returns = portfolio_returns - (txn_cost * turnover)
    
    mean_return = net_returns.mean()
    std_return = net_returns.std() + 1e-6 
    return -((mean_return / std_return) * math.sqrt(252))

def allocate_portfolio_discrete(continuous_signals, top_k=4):
    """
    COMPLIANT ALLOCATOR: Translates raw alphas strictly into 1 (Buy), 0 (Hold), or -1 (Sell).
    """
    batch_size = continuous_signals.shape[0]
    weights = torch.zeros_like(continuous_signals)
    
    for i in range(batch_size):
        alphas = continuous_signals[i]
        
        _, top_long_idx = torch.topk(alphas, top_k)
        _, top_short_idx = torch.topk(-alphas, top_k)
        
        for idx in top_long_idx:
            if alphas[idx] > 0.15: 
                weights[i, idx] = 1.0
                
        for idx in top_short_idx:
            if alphas[idx] < -0.15: 
                weights[i, idx] = -1.0

    active_exposure = torch.abs(weights).sum(dim=1, keepdim=True)
    cash_column = torch.where(active_exposure == 0, 1.0, 0.0)
    
    return torch.cat((cash_column, weights), dim=1)

def train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.001):
    base_optimizer = torch.optim.AdamW
    meta_optimizer = SAM(model.parameters(), base_optimizer, lr=meta_lr, weight_decay=1e-4, rho=0.05)
    inner_criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(meta_epochs):
        epoch_loss = 0.0
        batches = list(train_loader)
        
        # Process in pairs: Batch i for Adaptation, Batch i+1 for Evaluation
        for i in range(0, len(batches) - 1, 2):
            X_adapt, y_adapt = batches[i][0].to(device), batches[i][1].to(device)
            X_eval, y_eval = batches[i+1][0].to(device), batches[i+1][1].to(device)
            
            y_adapt = torch.nan_to_num(y_adapt, nan=0.0)
            y_eval = torch.nan_to_num(y_eval, nan=0.0)
            
            # ==========================================
            # PASS 1: Base Fast Adaptation
            # ==========================================
            preds = model(X_adapt)
            inner_loss = inner_criterion(preds, y_adapt)
            
            # Calculate inner gradients (allow_unused=True prevents crashes on inner_lrs)
            grads = torch.autograd.grad(inner_loss, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    # Parameter not used in forward pass (e.g., inner_lrs itself)
                    adapted_weights[name] = param
                else:
                    # Apply the learnable inner learning rate for this specific layer
                    clean_name = name.replace('.', '_')
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
            
            meta_optimizer.zero_grad()
            
            # Evaluate using the newly adapted weights
            eval_preds = functional_call(model, adapted_weights, X_eval) 
            meta_loss = differentiable_sharpe_loss(eval_preds, y_eval, txn_cost=0.1)
            
            # Backpropagate through the entire adaptation process
            meta_loss.backward() 
            
            # Shift global weights to the worst-case neighborhood
            meta_optimizer.first_step(zero_grad=True)
            
            # ==========================================
            # PASS 2: Worst-Case Fast Adaptation
            # ==========================================
            # Because SAM moved the base weights, we must re-adapt from this new starting point
            preds_adv = model(X_adapt)
            inner_loss_adv = inner_criterion(preds_adv, y_adapt)
            grads_adv = torch.autograd.grad(inner_loss_adv, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights_adv = {}
            for (name, param), grad in zip(model.named_parameters(), grads_adv):
                if grad is None:
                    adapted_weights_adv[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights_adv[name] = param - (model.inner_lrs[clean_name] * grad)
            
            # Evaluate the model in its worst-case adapted state
            eval_preds_perturbed = functional_call(model, adapted_weights_adv, X_eval)
            meta_loss_perturbed = differentiable_sharpe_loss(eval_preds_perturbed, y_eval, txn_cost=0.1)
            
            # Calculate final gradients and update the original global weights
            meta_loss_perturbed.backward()
            meta_optimizer.second_step(zero_grad=True)
            
            epoch_loss += meta_loss_perturbed.item()
            
        num_task_batches = len(batches) // 2
        avg_loss = (epoch_loss / num_task_batches) if num_task_batches > 0 else 0.0
        print(f"Meta-Epoch {epoch+1}/{meta_epochs} | Avg Worst-Case Sharpe Loss: {avg_loss:.4f}")
        
    return model


def execute_advanced_strategy(ohlcv_path, returns_path):
    print("Initializing Unrestricted Advanced Strategy Engine...")
    
    # --- THE PYTORCH 2.0 SDPA FIX ---
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    # --------------------------------
    
    client = Numin2API()
    
    # 1. Load Data using Custom OHLCV Pipeline
    print("Generating Custom Temporal Splits (70% Train, 15% Val, 15% Test)...")
    train_loader, val_loader, test_loader = get_custom_dataloaders(ohlcv_path, returns_path, batch_size=32)
    
    # 2. Instantiate the Meta-SGD Architecture
    model = MetaSGDTransformer(
        num_assets=50, 
        d_model=256, 
        nhead=8, 
        num_layers=4, 
        dropout=0.3
    ).to(device)
    
    # 3. Meta-Training Phase
    print(f"\n{'='*55}")
    print("--- Phase 1: Meta-Training ---")
    print(f"{'='*55}")
    model = train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.001)
    
    # 4. Testing & Signal Generation Phase
    print(f"\n{'='*55}")
    print("--- Phase 2: Out-of-Sample Testing & Backtest ---")
    print(f"{'='*55}")
    
    model.eval()
    all_test_positions = []
    all_test_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            
            # Generate raw alpha scores [-1, 1]
            continuous_predictions = model(X_batch)
            
            # FIXED: Call the compliant discrete allocator! (No temperature argument needed)
            discrete_batch_positions = allocate_portfolio_discrete(
                continuous_predictions, 
                top_k=4
            )
            
            # Sanitize targets
            nan_mask = torch.isnan(y_batch).to(device)
            discrete_batch_positions[:, 1:][nan_mask] = 0.0 
            clean_y_batch = torch.nan_to_num(y_batch, nan=0.0)
            
            all_test_positions.append(discrete_batch_positions.cpu())
            all_test_targets.append(clean_y_batch.cpu())
            
    full_test_positions = torch.cat(all_test_positions, dim=0)
    full_test_targets = torch.cat(all_test_targets, dim=0)
    
    print(f">> Submitting {full_test_positions.shape[0]} trading days to Numin2 Backtester...")
    
    # 5. Execute API Backtest
    # FIXED: Change txn_cost to realistic 0.001 (10 basis points)
    results = client.backtest_positions(
        positions=full_test_positions.numpy(), 
        targets=full_test_targets.numpy(), 
        txn_cost=0.1
    )
    
    print(f"\n>> RAW BACKTEST OUTPUT:")
    print(results)
    
    return results

if __name__ == "__main__":
    # Ensure your global paths match your local environment
    OHLCV_PATH = '../data/consolidated_daily_ohlcv.parquet'
    RETURNS_PATH = '../data/consolidated_daily_returns.parquet'
    
    execute_advanced_strategy(OHLCV_PATH, RETURNS_PATH)

## Experiment 3: online adaptation + "lazy" stat-arb allocator

* Inputs: raw return plus cross-sectionally z-scored Garman-Klass volatility, relative volume,
  return and 60-day momentum; 252-day windows.
* Meta-training: Meta-SGD + SAM on the turnover-aware Sharpe loss.
* Test time: online adaptation, then an always-invested top-2 long / top-2 short allocator with
  an inertia bonus of 1.5 to reduce turnover.

In [ ]:
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F  # <--- Add this line right here
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

# --- API Imports ---
from numin2 import Numin2API, NuminDataset, get_range_dataloader, get_month_dataloader
from torch.func import functional_call
# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')
DATA_PATH = '/tmp/consolidated_daily_returns.parquet' 

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from torch.utils.data import Dataset, DataLoader

import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

from numin2 import Numin2API

# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')

# Setup device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Custom Dataloader Pipeline
# ==========================================
class NuminAdvancedDataset(Dataset):
    # CHANGE: Increased seq_len from 60 to 252
    def __init__(self, ohlcv_path, returns_path, seq_len=252): 
        print("Loading and aligning Parquet files...")
        df_ohlcv = pd.read_parquet(ohlcv_path)
        df_returns = pd.read_parquet(returns_path)
        
        # 1. Flatten OHLCV if it has MultiIndex Columns
        if isinstance(df_ohlcv.columns, pd.MultiIndex):
            df_ohlcv = df_ohlcv.stack(level=1, future_stack=True).reset_index()
            df_ohlcv.rename(columns={df_ohlcv.columns[0]: 'date', df_ohlcv.columns[1]: 'asset'}, inplace=True)
        elif isinstance(df_ohlcv.index, pd.MultiIndex):
            df_ohlcv = df_ohlcv.reset_index()
            
        # 2. Flatten Returns if it's a wide flat table
        if not isinstance(df_returns.columns, pd.MultiIndex) and 'asset' not in [str(c).lower() for c in df_returns.columns]:
            df_returns.index.name = 'date'
            df_returns = df_returns.reset_index().melt(id_vars=['date'], var_name='asset', value_name='returns')
        elif isinstance(df_returns.index, pd.MultiIndex):
            df_returns = df_returns.reset_index()

        # 3. Standardize column names
        df_ohlcv.columns = [str(c).lower() for c in df_ohlcv.columns]
        df_returns.columns = [str(c).lower() for c in df_returns.columns]
        
        if 'returns' in df_ohlcv.columns and 'returns' in df_returns.columns:
            df_ohlcv.drop(columns=['returns'], inplace=True)
            
        print("Schema flattened. Executing merge...")
        df_merged = pd.merge(df_ohlcv, df_returns, on=['date', 'asset'], how='inner')
        df_merged.sort_values(by=['date', 'asset'], inplace=True)
        
        self.dates = df_merged['date'].unique()
        self.num_assets = len(df_merged['asset'].unique()) 
        
        feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']
        
        self.data = np.zeros((len(self.dates), self.num_assets, 6))
        
        for i, f in enumerate(feature_cols):
            if f in df_merged.columns:
                pivot_df = df_merged.pivot(index='date', columns='asset', values=f).fillna(0)
                self.data[:, :, i] = pivot_df.values
            
        self.data = torch.FloatTensor(self.data)
        self.seq_len = seq_len
        print(f"Data successfully compiled! Tensor Shape: {self.data.shape}")

    def __len__(self):
        return len(self.dates) - self.seq_len

    def __getitem__(self, idx):
        X = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len, :, 5] 
        return X, y

def get_custom_dataloaders(ohlcv_path, returns_path, batch_size=32):
    full_dataset = NuminAdvancedDataset(ohlcv_path, returns_path)
    # Temporal splitting as required by the assignment
    train_size = int(0.7 * len(full_dataset))
    val_size = int(0.15 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    
    train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
    val_dataset = torch.utils.data.Subset(full_dataset, range(train_size, train_size + val_size))
    test_dataset = torch.utils.data.Subset(full_dataset, range(train_size + val_size, len(full_dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

# ==========================================
# 2. Advanced Architecture
# ==========================================
class AdvancedQuantFeatureBlock(nn.Module):
    def __init__(self, num_assets=50):
        super(AdvancedQuantFeatureBlock, self).__init__()
        self.num_assets = num_assets
        self.out_features = num_assets * 5

    def forward(self, x):
        B, T, N, F_dim = x.shape
        O, H, L, C = x[..., 0], x[..., 1], x[..., 2], x[..., 3]
        V, R = x[..., 4], x[..., 5]
        
        # Helper: Cross-Sectional Normalizer
        def cs_zscore(tensor):
            mean = tensor.mean(dim=-1, keepdim=True)
            std = tensor.std(dim=-1, keepdim=True) + 1e-6
            return (tensor - mean) / std

        # 1. GK Volatility (Cross-Sectionally Normalized)
        log_hl = torch.log(H / (L + 1e-6) + 1e-6)
        log_co = torch.log(C / (O + 1e-6) + 1e-6)
        gk_internal = 0.5 * log_hl**2 - (2 * math.log(2) - 1) * log_co**2
        gk_vol = torch.sqrt(torch.clamp(gk_internal, min=0.0))
        gk_vol_z = cs_zscore(gk_vol).unsqueeze(-1)
        
        # 2. Relative Volume (Cross-Sectionally Normalized)
        cumsum_v = torch.cumsum(V, dim=1)
        sma_v = torch.zeros_like(V)
        sma_v[:, :20, :] = cumsum_v[:, :20, :] / torch.arange(1, min(T, 20)+1, device=x.device).view(1, -1, 1)
        if T > 20:
            sma_v[:, 20:, :] = (cumsum_v[:, 20:, :] - cumsum_v[:, :-20, :]) / 20.0
        rel_vol = V / (sma_v + 1e-6)
        rel_vol_z = cs_zscore(rel_vol).unsqueeze(-1)
        
        # 3. Daily Returns (Cross-Sectionally Normalized)
        r_z = cs_zscore(R).unsqueeze(-1)
        
        # 4. 60-Day Momentum (Cross-Sectionally Normalized)
        cumsum_r = torch.cumsum(R, dim=1)
        mom_60d = torch.zeros_like(R)
        mom_60d[:, :60, :] = cumsum_r[:, :60, :]
        if T > 60:
            mom_60d[:, 60:, :] = cumsum_r[:, 60:, :] - cumsum_r[:, :-60, :]
        mom_60d_z = cs_zscore(mom_60d).unsqueeze(-1)
        
        # Concatenate robust relative features
        out = torch.cat([R.unsqueeze(-1), gk_vol_z, rel_vol_z, r_z, mom_60d_z], dim=-1)
        return torch.nan_to_num(out.view(B, T, -1), nan=0.0)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.4, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MetaSGDTransformer(nn.Module):
    def __init__(self, num_assets=50, d_model=256, nhead=8, num_layers=4, dropout=0.3):
        super(MetaSGDTransformer, self).__init__()
        self.feature_block = AdvancedQuantFeatureBlock(num_assets=num_assets)
        self.input_projection = nn.Linear(self.feature_block.out_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, 128), nn.LayerNorm(128), nn.GELU(), 
            nn.Dropout(dropout), nn.Linear(128, num_assets)
        )
        
        self.inner_lrs = nn.ParameterDict({
            name.replace('.', '_'): nn.Parameter(torch.ones_like(param) * 0.005)
            for name, param in self.named_parameters() if param.requires_grad
        })

    def forward(self, x):
        x = self.feature_block(x)
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        encoded = self.transformer(x)
        raw_alpha = self.output_projection(encoded[:, -1, :])
        return torch.tanh(raw_alpha)

# ==========================================
# 3. Meta-Optimizers & Losses
# ==========================================
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  
        self.base_optimizer.step()  
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]),
            p=2
        )
        return norm

def differentiable_sharpe_loss(predictions, targets, txn_cost=0.1):
    portfolio_returns = (predictions * targets).sum(dim=1) 
    cash_start = torch.zeros(1, predictions.size(1), device=predictions.device)
    prev_predictions = torch.cat([cash_start, predictions[:-1]], dim=0)
    
    turnover = torch.abs(predictions - prev_predictions).sum(dim=1)
    net_returns = portfolio_returns - (txn_cost * turnover)
    
    mean_return = net_returns.mean()
    std_return = net_returns.std() + 1e-6 
    return -((mean_return / std_return) * math.sqrt(252))

def allocate_portfolio_lazy_statarb(continuous_signals, prev_weights=None, top_k=4, inertia_bonus=1.5):
    """
    THE FINAL ALLOCATOR:
    Combines Dollar-Neutral StatArb (Top K / Bottom K) with Extreme Inertia
    to harvest the relative spread without bleeding to the 0.1 transaction cost.
    """
    batch_size = continuous_signals.shape[0]
    num_assets = continuous_signals.shape[1]
    weights = torch.zeros_like(continuous_signals)
    
    if prev_weights is None:
        prev_weights = torch.zeros(num_assets, device=continuous_signals.device)
        
    for i in range(batch_size):
        alphas = continuous_signals[i]
        
        # Apply the 1.5 inertia boost to physically prevent the model from rotating 
        # unless a new stock exhibits an overwhelmingly strong relative signal.
        adjusted_alphas = alphas + (prev_weights * inertia_bonus)
        
        _, top_long_idx = torch.topk(adjusted_alphas, top_k)
        _, top_short_idx = torch.topk(-adjusted_alphas, top_k)
        
        current_w = torch.zeros(num_assets, device=continuous_signals.device)
        current_w[top_long_idx] = 1.0
        
        for idx in top_short_idx:
            if current_w[idx] != 1.0: 
                current_w[idx] = -1.0

        weights[i] = current_w
        prev_weights = current_w 

    active_exposure = torch.abs(weights).sum(dim=1, keepdim=True)
    cash_column = torch.where(active_exposure == 0, 1.0, 0.0)
    
    return torch.cat((cash_column, weights), dim=1), prev_weights

def train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.0003):
    base_optimizer = torch.optim.AdamW
    meta_optimizer = SAM(model.parameters(), base_optimizer, lr=meta_lr, weight_decay=1e-4, rho=0.05)
    inner_criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(meta_epochs):
        epoch_loss = 0.0
        batches = list(train_loader)
        
        # Process in pairs: Batch i for Adaptation, Batch i+1 for Evaluation
        for i in range(0, len(batches) - 1, 2):
            X_adapt, y_adapt = batches[i][0].to(device), batches[i][1].to(device)
            X_eval, y_eval = batches[i+1][0].to(device), batches[i+1][1].to(device)
            
            y_adapt = torch.nan_to_num(y_adapt, nan=0.0)
            y_eval = torch.nan_to_num(y_eval, nan=0.0)
            
            # ==========================================
            # PASS 1: Base Fast Adaptation
            # ==========================================
            preds = model(X_adapt)
            inner_loss = inner_criterion(preds, y_adapt)
            
            # Calculate inner gradients
            grads = torch.autograd.grad(inner_loss, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    adapted_weights[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
            
            meta_optimizer.zero_grad()
            
            # Evaluate using the newly adapted weights (PURE SHARPE LOSS, NO SPARSITY)
            eval_preds = functional_call(model, adapted_weights, X_eval) 
            meta_loss = differentiable_sharpe_loss(eval_preds, y_eval, txn_cost=0.1)
            
            meta_loss.backward() 
            meta_optimizer.first_step(zero_grad=True)
            
            # ==========================================
            # PASS 2: Worst-Case Fast Adaptation
            # ==========================================
            preds_adv = model(X_adapt)
            inner_loss_adv = inner_criterion(preds_adv, y_adapt)
            grads_adv = torch.autograd.grad(inner_loss_adv, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights_adv = {}
            for (name, param), grad in zip(model.named_parameters(), grads_adv):
                if grad is None:
                    adapted_weights_adv[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights_adv[name] = param - (model.inner_lrs[clean_name] * grad)
            
            # Evaluate the model in its worst-case adapted state (PURE SHARPE LOSS, NO SPARSITY)
            eval_preds_perturbed = functional_call(model, adapted_weights_adv, X_eval)
            meta_loss_perturbed = differentiable_sharpe_loss(eval_preds_perturbed, y_eval, txn_cost=0.1)
            
            meta_loss_perturbed.backward()
            meta_optimizer.second_step(zero_grad=True)
            
            epoch_loss += meta_loss_perturbed.item()
            
        num_task_batches = len(batches) // 2
        avg_loss = (epoch_loss / num_task_batches) if num_task_batches > 0 else 0.0
        print(f"Meta-Epoch {epoch+1}/{meta_epochs} | Avg Worst-Case Sharpe Loss: {avg_loss:.4f}")
        
    return model

def allocate_portfolio_statarb(continuous_signals, top_k=4):
    """
    STAT-ARB DOLLAR-NEUTRAL ALLOCATOR
    Always longs the Top K stocks and shorts the Bottom K stocks, regardless of absolute magnitude.
    Captures the "relatively higher / relatively lower" spread described in the assignment.
    """
    batch_size = continuous_signals.shape[0]
    num_assets = continuous_signals.shape[1]
    weights = torch.zeros_like(continuous_signals)
    
    for i in range(batch_size):
        alphas = continuous_signals[i]
        
        # Rank all 50 stocks strictly relative to each other on that specific day
        # Top K get +1.0 (Buy)
        _, top_long_idx = torch.topk(alphas, top_k)
        weights[i, top_long_idx] = 1.0
        
        # Bottom K get -1.0 (Sell)
        _, top_short_idx = torch.topk(-alphas, top_k)
        weights[i, top_short_idx] = -1.0

    active_exposure = torch.abs(weights).sum(dim=1, keepdim=True)
    cash_column = torch.where(active_exposure == 0, 1.0, 0.0)
    
    # We do not pass state back; StatArb relies on daily relative rebalancing
    return torch.cat((cash_column, weights), dim=1)

def execute_advanced_strategy(ohlcv_path, returns_path):
    print("Initializing Unrestricted Advanced Strategy Engine...")
    
    # --- THE PYTORCH 2.0 SDPA FIX ---
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    # --------------------------------
    
    client = Numin2API()
    
    # 1. Load Data using Custom OHLCV Pipeline
    print("Generating Custom Temporal Splits (70% Train, 15% Val, 15% Test)...")
    train_loader, val_loader, test_loader = get_custom_dataloaders(ohlcv_path, returns_path, batch_size=128)
    
    # 2. Instantiate the Meta-SGD Architecture
    model = MetaSGDTransformer(
        num_assets=50, 
        d_model=256, 
        nhead=8, 
        num_layers=4, 
        dropout=0.3
    ).to(device)
    
    # 3. Meta-Training Phase
    print(f"\n{'='*55}")
    print("--- Phase 1: Meta-Training ---")
    print(f"{'='*55}")
    model = train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.0003)
    
    # 4. Testing & Signal Generation Phase
    print(f"\n{'='*55}")
    print("--- Phase 2: Out-of-Sample Testing (ONLINE ADAPTATION) ---")
    print(f"{'='*55}")
    
    all_test_positions = []
    all_test_targets = []
    current_portfolio_state = None 
    
    X_prev, y_prev = None, None
    inner_criterion = nn.MSELoss()
    
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        clean_y_batch = torch.nan_to_num(y_batch, nan=0.0).to(device)
        
        # ==========================================
        # STEP A: Fast-Adaptation to the Recent Regime
        # ==========================================
        if X_prev is not None:
            model.train() 
            preds_prev = model(X_prev)
            loss_prev = inner_criterion(preds_prev, y_prev)
            
            grads = torch.autograd.grad(loss_prev, model.parameters(), retain_graph=False, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    adapted_weights[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
        else:
            adapted_weights = {name: param for name, param in model.named_parameters()}
            
        # ==========================================
        # STEP B: Inference on the Current Regime
        # ==========================================
        
        
        with torch.no_grad():
            model.eval() 
            
            continuous_predictions = functional_call(model, adapted_weights, X_batch)
            
            # CONCENTRATED STAT-ARB: Reduce fee exposure by only taking the best 2 spreads
            discrete_batch_positions, current_portfolio_state = allocate_portfolio_lazy_statarb(
                continuous_predictions, 
                prev_weights=current_portfolio_state,
                top_k=2,             # REDUCED: Maximize spread, minimize fee multiplication
                inertia_bonus=1.5    # Maintain extreme defense against the 0.1 fee
            )
            
        nan_mask = torch.isnan(y_batch).to(device)
        discrete_batch_positions[:, 1:][nan_mask] = 0.0 
        clean_y_batch = torch.nan_to_num(y_batch, nan=0.0).to(device)
        
        all_test_positions.append(discrete_batch_positions.cpu())
        all_test_targets.append(clean_y_batch.cpu())
        
        # Shift the rolling window forward in time
        X_prev = X_batch
        y_prev = clean_y_batch
        
    full_test_positions = torch.cat(all_test_positions, dim=0)
    full_test_targets = torch.cat(all_test_targets, dim=0)
    
    print(f">> Submitting {full_test_positions.shape[0]} trading days to Numin2 Backtester...")
    
    # Executing with the brutal 10% constraint
    results = client.backtest_positions(
        positions=full_test_positions.numpy(), 
        targets=full_test_targets.numpy(), 
        txn_cost=0.1  
    )
    
    print(f"\n>> RAW BACKTEST OUTPUT:")
    print(results)
    
    return results

if __name__ == "__main__":
    # Ensure your global paths match your local environment
    OHLCV_PATH = '../data/consolidated_daily_ohlcv.parquet'
    RETURNS_PATH = '../data/consolidated_daily_returns.parquet'
    
    execute_advanced_strategy(OHLCV_PATH, RETURNS_PATH)

## Experiment 4 (final): Meta-SGD + SAM + L1 sparsity + online adaptation

* Inputs: raw return, Garman-Klass volatility, relative volume (vs. 20-day average),
  cross-sectional z-score of returns and 60-day momentum; 252-day windows, batch size 128.
* Meta-training: 10 epochs of Meta-SGD + SAM (`rho=0.05`, AdamW `lr=3e-4`) on the turnover-aware
  Sharpe loss (`txn_cost=0.1`) plus an L1 sparsity penalty.
* Test time: one adaptation step on the previous block using the learned inner learning rates,
  then the Absolute Conviction allocator (`top_k=2`, `enter_threshold=0.6`, `inertia=0.4`) and the
  Numin2 backtester with `txn_cost=0.1`.

In [ ]:
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F  # <--- Add this line right here
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

# --- API Imports ---
from numin2 import Numin2API, NuminDataset, get_range_dataloader, get_month_dataloader
from torch.func import functional_call
# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')
DATA_PATH = '/tmp/consolidated_daily_returns.parquet' 

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from torch.utils.data import Dataset, DataLoader

import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.func import functional_call
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader

from numin2 import Numin2API

# --- Prevent PyArrow Memory Crash ---
pd.set_option('io.parquet.engine', 'fastparquet')

# Setup device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. Custom Dataloader Pipeline
# ==========================================
class NuminAdvancedDataset(Dataset):
    # CHANGE: Increased seq_len from 60 to 252
    def __init__(self, ohlcv_path, returns_path, seq_len=252): 
        print("Loading and aligning Parquet files...")
        df_ohlcv = pd.read_parquet(ohlcv_path)
        df_returns = pd.read_parquet(returns_path)
        
        # 1. Flatten OHLCV if it has MultiIndex Columns
        if isinstance(df_ohlcv.columns, pd.MultiIndex):
            df_ohlcv = df_ohlcv.stack(level=1, future_stack=True).reset_index()
            df_ohlcv.rename(columns={df_ohlcv.columns[0]: 'date', df_ohlcv.columns[1]: 'asset'}, inplace=True)
        elif isinstance(df_ohlcv.index, pd.MultiIndex):
            df_ohlcv = df_ohlcv.reset_index()
            
        # 2. Flatten Returns if it's a wide flat table
        if not isinstance(df_returns.columns, pd.MultiIndex) and 'asset' not in [str(c).lower() for c in df_returns.columns]:
            df_returns.index.name = 'date'
            df_returns = df_returns.reset_index().melt(id_vars=['date'], var_name='asset', value_name='returns')
        elif isinstance(df_returns.index, pd.MultiIndex):
            df_returns = df_returns.reset_index()

        # 3. Standardize column names
        df_ohlcv.columns = [str(c).lower() for c in df_ohlcv.columns]
        df_returns.columns = [str(c).lower() for c in df_returns.columns]
        
        if 'returns' in df_ohlcv.columns and 'returns' in df_returns.columns:
            df_ohlcv.drop(columns=['returns'], inplace=True)
            
        print("Schema flattened. Executing merge...")
        df_merged = pd.merge(df_ohlcv, df_returns, on=['date', 'asset'], how='inner')
        df_merged.sort_values(by=['date', 'asset'], inplace=True)
        
        self.dates = df_merged['date'].unique()
        self.num_assets = len(df_merged['asset'].unique()) 
        
        feature_cols = ['open', 'high', 'low', 'close', 'volume', 'returns']
        
        self.data = np.zeros((len(self.dates), self.num_assets, 6))
        
        for i, f in enumerate(feature_cols):
            if f in df_merged.columns:
                pivot_df = df_merged.pivot(index='date', columns='asset', values=f).fillna(0)
                self.data[:, :, i] = pivot_df.values
            
        self.data = torch.FloatTensor(self.data)
        self.seq_len = seq_len
        print(f"Data successfully compiled! Tensor Shape: {self.data.shape}")

    def __len__(self):
        return len(self.dates) - self.seq_len

    def __getitem__(self, idx):
        X = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len, :, 5] 
        return X, y

def get_custom_dataloaders(ohlcv_path, returns_path, batch_size=32):
    full_dataset = NuminAdvancedDataset(ohlcv_path, returns_path)
    # Temporal splitting as required by the assignment
    train_size = int(0.7 * len(full_dataset))
    val_size = int(0.15 * len(full_dataset))
    test_size = len(full_dataset) - train_size - val_size
    
    train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
    val_dataset = torch.utils.data.Subset(full_dataset, range(train_size, train_size + val_size))
    test_dataset = torch.utils.data.Subset(full_dataset, range(train_size + val_size, len(full_dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader

# ==========================================
# 2. Advanced Architecture
# ==========================================
class AdvancedQuantFeatureBlock(nn.Module):
    def __init__(self, num_assets=50):
        super(AdvancedQuantFeatureBlock, self).__init__()
        self.num_assets = num_assets
        # Updated out_features: 5 features per asset (Returns, GK Vol, Rel Vol, Z-Score, Mom)
        self.out_features = num_assets * 5 

    def forward(self, x):
        B, T, N, F_dim = x.shape
        O, H, L, C = x[..., 0], x[..., 1], x[..., 2], x[..., 3]
        V, R = x[..., 4], x[..., 5]
        
        # 1. Garman-Klass Volatility Proxy
        log_hl = torch.log(H / (L + 1e-6) + 1e-6)
        log_co = torch.log(C / (O + 1e-6) + 1e-6)
        gk_internal = 0.5 * log_hl**2 - (2 * math.log(2) - 1) * log_co**2
        gk_vol = torch.sqrt(torch.clamp(gk_internal, min=0.0)).unsqueeze(-1)
        
        # 2. Relative Volume
        cumsum_v = torch.cumsum(V, dim=1)
        sma_v = torch.zeros_like(V)
        sma_v[:, :20, :] = cumsum_v[:, :20, :] / torch.arange(1, min(T, 20)+1, device=x.device).view(1, -1, 1)
        if T > 20:
            sma_v[:, 20:, :] = (cumsum_v[:, 20:, :] - cumsum_v[:, :-20, :]) / 20.0
        rel_vol = (V / (sma_v + 1e-6)).unsqueeze(-1)
        
        # 3. Cross-Sectional Z-Score
        mean_r = R.mean(dim=-1, keepdim=True)
        std_r = R.std(dim=-1, keepdim=True) + 1e-6
        z_score = ((R - mean_r) / std_r).unsqueeze(-1)
        
        cumsum_r = torch.cumsum(R, dim=1)
        mom_60d = torch.zeros_like(R)
        mom_60d[:, :60, :] = cumsum_r[:, :60, :]
        if T > 60:
            mom_60d[:, 60:, :] = cumsum_r[:, 60:, :] - cumsum_r[:, :-60, :]
        
        # Normalize the macro trend so it plays nicely with the other features
        mom_60d = (mom_60d / 60.0).unsqueeze(-1) 
        
        # Concatenate using the new macro feature
        out = torch.cat([R.unsqueeze(-1), gk_vol, rel_vol, z_score, mom_60d], dim=-1)
        return torch.nan_to_num(out.view(B, T, -1), nan=0.0)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.4, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class MetaSGDTransformer(nn.Module):
    def __init__(self, num_assets=50, d_model=256, nhead=8, num_layers=4, dropout=0.3):
        super(MetaSGDTransformer, self).__init__()
        self.feature_block = AdvancedQuantFeatureBlock(num_assets=num_assets)
        self.input_projection = nn.Linear(self.feature_block.out_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, 128), nn.LayerNorm(128), nn.GELU(), 
            nn.Dropout(dropout), nn.Linear(128, num_assets)
        )
        
        self.inner_lrs = nn.ParameterDict({
            name.replace('.', '_'): nn.Parameter(torch.ones_like(param) * 0.005)
            for name, param in self.named_parameters() if param.requires_grad
        })

    def forward(self, x):
        x = self.feature_block(x)
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        encoded = self.transformer(x)
        raw_alpha = self.output_projection(encoded[:, -1, :])
        return torch.tanh(raw_alpha)

# ==========================================
# 3. Meta-Optimizers & Losses
# ==========================================
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  
        self.base_optimizer.step()  
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]),
            p=2
        )
        return norm

def differentiable_sharpe_loss(predictions, targets, txn_cost=0.1):
    portfolio_returns = (predictions * targets).sum(dim=1) 
    cash_start = torch.zeros(1, predictions.size(1), device=predictions.device)
    prev_predictions = torch.cat([cash_start, predictions[:-1]], dim=0)
    
    turnover = torch.abs(predictions - prev_predictions).sum(dim=1)
    net_returns = portfolio_returns - (txn_cost * turnover)
    
    mean_return = net_returns.mean()
    std_return = net_returns.std() + 1e-6 
    return -((mean_return / std_return) * math.sqrt(252))

def allocate_portfolio_discrete(continuous_signals, top_k=4, confidence_z_score=1.0):
    """
    COMPLIANT ALLOCATOR: Translates raw alphas strictly into 1, 0, or -1.
    Uses cross-sectional Z-scoring to ensure we only trade with high relative conviction.
    """
    batch_size = continuous_signals.shape[0]
    weights = torch.zeros_like(continuous_signals)
    
    for i in range(batch_size):
        alphas = continuous_signals[i]
        
        # Z-score the alphas to find the relative conviction for the day
        mean_alpha = alphas.mean()
        std_alpha = alphas.std() + 1e-6
        z_alphas = (alphas - mean_alpha) / std_alpha
        
        _, top_long_idx = torch.topk(z_alphas, top_k)
        _, top_short_idx = torch.topk(-z_alphas, top_k)
        
        # Only buy if the signal is > 1 standard deviation above the daily mean
        for idx in top_long_idx:
            if z_alphas[idx] > confidence_z_score: 
                weights[i, idx] = 1.0
                
        # Only sell if the signal is > 1 standard deviation below the daily mean
        for idx in top_short_idx:
            if z_alphas[idx] < -confidence_z_score: 
                weights[i, idx] = -1.0

    active_exposure = torch.abs(weights).sum(dim=1, keepdim=True)
    cash_column = torch.where(active_exposure == 0, 1.0, 0.0)
    
    return torch.cat((cash_column, weights), dim=1)

def train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.001):
    base_optimizer = torch.optim.AdamW
    meta_optimizer = SAM(model.parameters(), base_optimizer, lr=meta_lr, weight_decay=1e-4, rho=0.05)
    inner_criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(meta_epochs):
        epoch_loss = 0.0
        batches = list(train_loader)
        
        # Process in pairs: Batch i for Adaptation, Batch i+1 for Evaluation
        for i in range(0, len(batches) - 1, 2):
            X_adapt, y_adapt = batches[i][0].to(device), batches[i][1].to(device)
            X_eval, y_eval = batches[i+1][0].to(device), batches[i+1][1].to(device)
            
            y_adapt = torch.nan_to_num(y_adapt, nan=0.0)
            y_eval = torch.nan_to_num(y_eval, nan=0.0)
            
            # ==========================================
            # PASS 1: Base Fast Adaptation
            # ==========================================
            preds = model(X_adapt)
            inner_loss = inner_criterion(preds, y_adapt)
            
            # Calculate inner gradients (allow_unused=True prevents crashes on inner_lrs)
            grads = torch.autograd.grad(inner_loss, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    # Parameter not used in forward pass (e.g., inner_lrs itself)
                    adapted_weights[name] = param
                else:
                    # Apply the learnable inner learning rate for this specific layer
                    clean_name = name.replace('.', '_')
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
            
            meta_optimizer.zero_grad()
            
            # Evaluate using the newly adapted weights
            eval_preds = functional_call(model, adapted_weights, X_eval) 
            base_sharpe_loss = differentiable_sharpe_loss(eval_preds, y_eval, txn_cost=0.1)
            
            # L1 SPARSITY: Penalize the network for any prediction that isn't exactly 0.0
            sparsity_penalty = 0.5 * torch.mean(torch.abs(eval_preds)) 
            meta_loss = base_sharpe_loss + sparsity_penalty
            
            meta_loss.backward() 
            meta_optimizer.first_step(zero_grad=True)
            # ==========================================
            # PASS 2: Worst-Case Fast Adaptation
            # ==========================================
            # Because SAM moved the base weights, we must re-adapt from this new starting point
            preds_adv = model(X_adapt)
            inner_loss_adv = inner_criterion(preds_adv, y_adapt)
            grads_adv = torch.autograd.grad(inner_loss_adv, model.parameters(), create_graph=True, allow_unused=True)
            
            adapted_weights_adv = {}
            for (name, param), grad in zip(model.named_parameters(), grads_adv):
                if grad is None:
                    adapted_weights_adv[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    adapted_weights_adv[name] = param - (model.inner_lrs[clean_name] * grad)
            
            # Evaluate the model in its worst-case adapted state
            eval_preds_perturbed = functional_call(model, adapted_weights_adv, X_eval)
            base_sharpe_loss_adv = differentiable_sharpe_loss(eval_preds_perturbed, y_eval, txn_cost=0.1)
            
            # Apply sparsity to the worst-case evaluation as well
            sparsity_penalty_adv = 0.5 * torch.mean(torch.abs(eval_preds_perturbed))
            meta_loss_perturbed = base_sharpe_loss_adv + sparsity_penalty_adv
            
            meta_loss_perturbed.backward()
            meta_optimizer.second_step(zero_grad=True)
            
            epoch_loss += meta_loss_perturbed.item()
            
        num_task_batches = len(batches) // 2
        avg_loss = (epoch_loss / num_task_batches) if num_task_batches > 0 else 0.0
        print(f"Meta-Epoch {epoch+1}/{meta_epochs} | Avg Worst-Case Sharpe Loss: {avg_loss:.4f}")
        
    return model

def allocate_portfolio_absolute(continuous_signals, prev_weights=None, top_k=2, enter_threshold=0.6, inertia=0.4):
    """
    ABSOLUTE SNIPER ALLOCATOR: 
    Relies on absolute tanh output magnitude, completely ignoring relative Z-scores.
    Defaults to 100% cash if the model lacks conviction.
    """
    batch_size = continuous_signals.shape[0]
    num_assets = continuous_signals.shape[1]
    weights = torch.zeros_like(continuous_signals)
    
    if prev_weights is None:
        prev_weights = torch.zeros(num_assets, device=continuous_signals.device)
        
    for i in range(batch_size):
        alphas = continuous_signals[i] # Raw tanh values [-1, 1]
        
        # INERTIA: If we hold a stock (+1), it gets a +0.4 boost, making it easier to hold 
        # than it was to enter. (0.6 to enter, but only needs to stay > 0.2 to hold).
        adjusted_alphas = alphas + (prev_weights * inertia)
        
        # Find the absolute top candidates
        long_scores, top_long_idx = torch.topk(adjusted_alphas, top_k)
        short_scores, top_short_idx = torch.topk(-adjusted_alphas, top_k)
        
        current_w = torch.zeros(num_assets, device=continuous_signals.device)
        
        # REQUIREMENT: The raw signal must beat the absolute threshold.
        # If the model is uncertain and outputs 0.1, it gets ignored. Cash is retained.
        for idx, score in zip(top_long_idx, long_scores):
            if score > enter_threshold: 
                current_w[idx] = 1.0
                
        for idx, score in zip(top_short_idx, short_scores):
            if score > enter_threshold: 
                current_w[idx] = -1.0

        weights[i] = current_w
        prev_weights = current_w 

    active_exposure = torch.abs(weights).sum(dim=1, keepdim=True)
    cash_column = torch.where(active_exposure == 0, 1.0, 0.0)
    
    return torch.cat((cash_column, weights), dim=1), prev_weights


def execute_advanced_strategy(ohlcv_path, returns_path):
    print("Initializing Unrestricted Advanced Strategy Engine...")
    
    # --- THE PYTORCH 2.0 SDPA FIX ---
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    # --------------------------------
    
    client = Numin2API()
    
    # 1. Load Data using Custom OHLCV Pipeline
    print("Generating Custom Temporal Splits (70% Train, 15% Val, 15% Test)...")
    train_loader, val_loader, test_loader = get_custom_dataloaders(ohlcv_path, returns_path, batch_size=128)
    
    # 2. Instantiate the Meta-SGD Architecture
    model = MetaSGDTransformer(
        num_assets=50, 
        d_model=256, 
        nhead=8, 
        num_layers=4, 
        dropout=0.3
    ).to(device)
    
    # 3. Meta-Training Phase
    print(f"\n{'='*55}")
    print("--- Phase 1: Meta-Training ---")
    print(f"{'='*55}")
    model = train_meta_sgd_sam(model, train_loader, meta_epochs=10, meta_lr=0.0003)
    
    # 4. Testing & Signal Generation Phase
    print(f"\n{'='*55}")
    print("--- Phase 2: Out-of-Sample Testing (ONLINE ADAPTATION) ---")
    print(f"{'='*55}")
    
    all_test_positions = []
    all_test_targets = []
    current_portfolio_state = None 
    
    # State trackers for the rolling adaptation window
    X_prev, y_prev = None, None
    inner_criterion = nn.MSELoss()
    
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        clean_y_batch = torch.nan_to_num(y_batch, nan=0.0).to(device)
        
        # ==========================================
        # STEP A: Fast-Adaptation to the Recent Regime
        # ==========================================
        if X_prev is not None:
            # Must be in train mode to build the computation graph for gradients
            model.train() 
            preds_prev = model(X_prev)
            loss_prev = inner_criterion(preds_prev, y_prev)
            
            # Compute inner gradients safely without updating the global weights
            grads = torch.autograd.grad(loss_prev, model.parameters(), retain_graph=False, allow_unused=True)
            
            adapted_weights = {}
            for (name, param), grad in zip(model.named_parameters(), grads):
                if grad is None:
                    adapted_weights[name] = param
                else:
                    clean_name = name.replace('.', '_')
                    # Apply the learned inner learning rate to shift into the current regime
                    adapted_weights[name] = param - (model.inner_lrs[clean_name] * grad)
        else:
            # First batch: no history available yet, use base global weights
            adapted_weights = {name: param for name, param in model.named_parameters()}
            
        # ==========================================
        # STEP B: Inference on the Current Regime
        # ==========================================
        with torch.no_grad():
            model.eval() # Freeze dropout/batchnorm for pure inference
            
            # Predict using the newly ADAPTED weights, not the global weights!
            continuous_predictions = functional_call(model, adapted_weights, X_batch)
            
            # Route through the Absolute Conviction Allocator
            discrete_batch_positions, current_portfolio_state = allocate_portfolio_absolute(
                continuous_predictions, 
                prev_weights=current_portfolio_state,
                top_k=2,
                enter_threshold=0.6, 
                inertia=0.4
            )
            
        # Sanitize targets and append outputs
        nan_mask = torch.isnan(y_batch).to(device)
        discrete_batch_positions[:, 1:][nan_mask] = 0.0 
        
        all_test_positions.append(discrete_batch_positions.cpu())
        all_test_targets.append(clean_y_batch.cpu())
        
        # Shift the rolling window forward in time
        X_prev = X_batch
        y_prev = clean_y_batch
        
    full_test_positions = torch.cat(all_test_positions, dim=0)
    full_test_targets = torch.cat(all_test_targets, dim=0)
    
    print(f">> Submitting {full_test_positions.shape[0]} trading days to Numin2 Backtester...")
    
    results = client.backtest_positions(
        positions=full_test_positions.numpy(), 
        targets=full_test_targets.numpy(), 
        txn_cost=0.1
    )
    
    print(f"\n>> RAW BACKTEST OUTPUT:")
    print(results)
    
    return results

if __name__ == "__main__":
    # Ensure your global paths match your local environment
    OHLCV_PATH = '../data/consolidated_daily_ohlcv.parquet'
    RETURNS_PATH = '../data/consolidated_daily_returns.parquet'
    
    execute_advanced_strategy(OHLCV_PATH, RETURNS_PATH)